# Job Salary Estimator — Week 6 Synthesis Project

**Pipeline:** Same as "The Price is Right" (curation → preprocessing → evaluation → traditional ML → NN/LLM → fine-tuning), applied to **predicting job salary from job description**.

## Data
- **Source:** [lukebarousse/data_jobs](https://huggingface.co/datasets/lukebarousse/data_jobs) — data analytics job postings with `salary_year_avg`, job title, and description text.
- **Target:** Annual salary in **thousands of USD** (e.g. 120 = $120k) so the week6 evaluator's error scale is meaningful.

## Sections
1. **Step 1 — Data curation:** Load dataset, filter, build JobItem (compatible with pricer.evaluator).  
2. **Step 2 — Preprocessing:** Train/val/test split; optional LLM normalization.  
3. **Step 3 — Evaluation + traditional ML:** Baselines, Linear Regression, NLP+LR, Random Forest, XGBoost.  
4. **Step 4 — NN + frontier LLM:** Simple neural net and LiteLLM-based salary predictor.  
5. **Step 5 — Fine-tuning:** JSONL prep and OpenAI fine-tuning (optional, requires API).

---
## Step 1: Data Curation

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv(override=True)

# Use pricer's evaluator (expects items with .title, .price, .summary)
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
from pricer.evaluator import evaluate

In [ ]:
LITE_MODE = True  # True: smaller subset for fast runs; False: larger data

### JobItem: compatible with pricer.evaluator
We use a simple class with `.title`, `.price` (salary in thousands), and `.summary` (text for prediction). The evaluator expects these attributes.

In [ ]:
class JobItem:
    """One job posting: title, salary (in thousands USD), and text summary for prediction."""
    __slots__ = ("title", "price", "summary", "id")

    def __init__(self, title: str, salary_usd: float, summary: str, id_: int | None = None):
        self.title = title[:80] if len(title) > 80 else title
        # Store salary in thousands so evaluator error scale (~40, ~80) is meaningful
        self.price = salary_usd / 1000.0
        self.summary = summary or ""
        self.id = id_

    def __repr__(self):
        return f"<JobItem {self.title!r} = ${self.price:.1f}k>"

In [ ]:
# Curation constants
MIN_SALARY_USD = 20_000
MAX_SALARY_USD = 500_000
MIN_TEXT_LEN = 50
MAX_ITEMS_LITE = 22_000  # train + val + test total for lite mode

In [ ]:
def load_and_curate_jobs(lite: bool = True) -> list[JobItem]:
    """Load lukebarousse/data_jobs, filter, and return list of JobItems."""
    ds = load_dataset("lukebarousse/data_jobs", split="train", trust_remote_code=True)
    df = ds.to_pandas()

    # salary_year_avg is in USD; fallback to salary_hour_avg * 2080 if needed
    if "salary_year_avg" in df.columns:
        df["salary_usd"] = pd.to_numeric(df["salary_year_avg"], errors="coerce")
    else:
        df["salary_usd"] = None
    if df["salary_usd"].isna().all() and "salary_hour_avg" in df.columns:
        hr = pd.to_numeric(df["salary_hour_avg"], errors="coerce")
        df["salary_usd"] = hr * 2080

    df = df.dropna(subset=["salary_usd"])
    df = df[(df["salary_usd"] >= MIN_SALARY_USD) & (df["salary_usd"] <= MAX_SALARY_USD)]

    # data_jobs has job_title, job_title_short, company_name, job_skills (list)
    title_col = "job_title" if "job_title" in df.columns else "job_title_short"
    if title_col not in df.columns:
        title_col = df.columns[0]
    df["title"] = df[title_col].astype(str).fillna("")
    # Build summary from title + company + skills for prediction
    def to_text(x):
        if isinstance(x, list):
            return " ".join(str(i) for i in x)
        return str(x) if pd.notna(x) else ""
    df["company"] = df["company_name"].astype(str).fillna("") if "company_name" in df.columns else ""
    df["skills"] = df["job_skills"].map(to_text) if "job_skills" in df.columns else ""
    df["summary"] = (df["title"] + " " + df["company"] + " " + df["skills"]).str.strip()
    df = df[df["summary"].str.len() >= MIN_TEXT_LEN]

    items = []
    for i, row in df.iterrows():
        items.append(JobItem(title=row["title"], salary_usd=float(row["salary_usd"]), summary=row["summary"], id_=len(items)))

    if lite and len(items) > MAX_ITEMS_LITE:
        random.Random(42).shuffle(items)
        items = items[:MAX_ITEMS_LITE]
    return items

In [ ]:
all_jobs = load_and_curate_jobs(lite=LITE_MODE)
print(f"Curated {len(all_jobs):,} job items")
if all_jobs:
    print(all_jobs[0])

---
## Step 2: Preprocessing — Train / Val / Test Split

In [ ]:
def train_val_test_split(items: list[JobItem], val_ratio=0.05, test_ratio=0.05, seed=42):
    n = len(items)
    rng = random.Random(seed)
    idx = list(range(n))
    rng.shuffle(idx)
    n_test = max(1, int(n * test_ratio))
    n_val = max(1, int(n * val_ratio))
    n_train = n - n_val - n_test
    train = [items[i] for i in idx[:n_train]]
    val = [items[i] for i in idx[n_train : n_train + n_val]]
    test = [items[i] for i in idx[n_train + n_val :]]
    return train, val, test

In [ ]:
train, val, test = train_val_test_split(all_jobs)
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

---
## Step 3: Evaluation + Traditional ML

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
mean_salary_k = np.mean([j.price for j in train])

def constant_pricer(item):
    return mean_salary_k

evaluate(constant_pricer, test, size=min(200, len(test)))

In [ ]:
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X_train = vectorizer.fit_transform([j.summary for j in train])
y_train = np.array([j.price for j in train])

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

def linear_regression_pricer(item):
    X = vectorizer.transform([item.summary])
    return float(lr_model.predict(X)[0])

evaluate(linear_regression_pricer, test, size=min(200, len(test)))

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

def random_forest_pricer(item):
    X = vectorizer.transform([item.summary])
    return float(rf_model.predict(X)[0])

evaluate(random_forest_pricer, test, size=min(200, len(test)))

In [ ]:
xgb_model = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

def xgboost_pricer(item):
    X = vectorizer.transform([item.summary])
    return float(xgb_model.predict(X)[0])

evaluate(xgboost_pricer, test, size=min(200, len(test)))

---
## Step 4: Neural Network + Frontier LLM

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
nn_vectorizer = HashingVectorizer(n_features=2000, stop_words="english", binary=True)
X_nn = nn_vectorizer.fit_transform([j.summary for j in train]).toarray().astype(np.float32)
y_nn = np.array([j.price for j in train], dtype=np.float32).reshape(-1, 1)
y_mean, y_std = y_nn.mean(), y_nn.std()
if y_std < 1e-6:
    y_std = 1.0
y_nn = (y_nn - y_mean) / y_std

X_t = torch.tensor(X_nn)
y_t = torch.tensor(y_nn)
ds = TensorDataset(X_t, y_t)
loader = DataLoader(ds, batch_size=64, shuffle=True)

class SmallNN(nn.Module):
    def __init__(self, input_size=2000, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nn_model = SmallNN().to(device)
opt = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
for epoch in range(5):
    nn_model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = nn.functional.mse_loss(nn_model(xb), yb)
        loss.backward()
        opt.step()
    print(f"Epoch {epoch+1} loss: {loss.item():.4f}")

def nn_pricer(item):
    X = nn_vectorizer.transform([item.summary]).toarray().astype(np.float32)
    x = torch.tensor(X).to(device)
    nn_model.eval()
    with torch.no_grad():
        out = nn_model(x).cpu().numpy()[0, 0] * y_std + y_mean
    return float(out)

In [ ]:
evaluate(nn_pricer, test, size=min(200, len(test)))

### Frontier LLM (LiteLLM)
Requires `LITELLM_API_KEY` or provider keys in env. Uncomment and set model to run.

In [ ]:
# from litellm import completion
# SALARY_PROMPT = "Estimate the annual salary in USD for this job. Reply with only a number (e.g. 95000)."
# LLM_MODEL = "gpt-4o-mini"  # or groq/llama, etc.
#
# def llm_salary_pricer(item):
#     resp = completion(model=LLM_MODEL, messages=[{"role": "user", "content": SALARY_PROMPT + "\n\n" + item.summary[:2000]}])
#     raw = resp.choices[0].message.content or "0"
#     import re
#     m = re.search(r"[\d,]+", raw.replace(",", ""))
#     val = float(m.group()) if m else 0
#     return val / 1000.0  # convert to thousands for evaluator
#
# evaluate(llm_salary_pricer, test, size=min(50, len(test)))

---
## Step 5: Fine-tuning (Optional)
Prepare JSONL for OpenAI fine-tuning. Run only if you have API access and want to fine-tune a small model for salary prediction.

In [ ]:
import json

def messages_for_job(item: JobItem):
    return [
        {"role": "user", "content": "Estimate annual salary in USD. Reply with only the number.\n\n" + item.summary[:2000]},
        {"role": "assistant", "content": str(int(item.price * 1000))}
    ]

def write_jsonl(items: list[JobItem], path: str, n_max=100):
    with open(path, "w") as f:
        for item in items[:n_max]:
            msg = messages_for_job(item)
            f.write(json.dumps({"messages": msg}) + "\n")

# write_jsonl(train, "fine_tune_train.jsonl", n_max=100)
# write_jsonl(val, "fine_tune_validation.jsonl", n_max=50)
# Then: openai.files.create(file=open("fine_tune_train.jsonl", "rb"), purpose="fine-tune"), etc.

---
## Results Summary
Run all cells above and compare evaluator output (average error in **thousands of USD**, e.g. 15 = $15k MAE). Lower is better. Optionally log constant_pricer, linear_regression_pricer, random_forest_pricer, xgboost_pricer, nn_pricer, and llm_salary_pricer in a small results table.